In [1]:
import os
import win32com.client as win32
import openpyxl

def insert_images_into_excel(excel_path, images_folder):
    """
    Insert images into matching Excel sheets.
    Each image must be named as: <sheet_name>.png
    
    Args:
        excel_path (str): Full path to Excel workbook.
        images_folder (str): Folder containing the .png images.
    """

    # Validate paths
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"Excel file not found:\n{excel_path}")

    if not os.path.exists(images_folder):
        raise FileNotFoundError(f"Images folder not found:\n{images_folder}")

    # Load workbook to get sheet names
    wb = openpyxl.load_workbook(excel_path, data_only=True)
    sheet_names = wb.sheetnames
    wb.close()

    # Start Excel COM
    excel_app = win32.Dispatch("Excel.Application")
    excel_app.Visible = False
    workbook = excel_app.Workbooks.Open(excel_path)

    try:
        for sheet_name in sheet_names:
            if sheet_name == "Summary":
                continue  # skip summary

            image_path = os.path.join(images_folder, f"{sheet_name}.png")

            if not os.path.exists(image_path):
                print(f"[SKIPPED] Image not found for sheet: {sheet_name}")
                continue

            try:
                xl_sheet = workbook.Sheets(sheet_name)
                xl_sheet.Activate()

                # Column location logic
                if sheet_name.endswith("_LL"):
                    left_col = 16
                else:
                    left_col = 14

                # Insert image
                xl_sheet.Shapes.AddPicture(
                    Filename=os.path.abspath(image_path),
                    LinkToFile=False,
                    SaveWithDocument=True,
                    Left=xl_sheet.Cells(2, left_col).Left,
                    Top=xl_sheet.Cells(2, left_col).Top,
                    Width=800,
                    Height=400
                )

                print(f"[OK] Inserted image into sheet: {sheet_name}")

            except Exception as e:
                print(f"[ERROR] Failed inserting into {sheet_name}: {e}")

        workbook.Save()
        print("\n✔ All available images inserted successfully!")

    finally:
        workbook.Close()
        excel_app.Quit()


In [3]:
excel_file = r"C:\Users\Raghavendra\Downloads\WCR VB Braking Trials\5001_VB_16_Coaches_DN_Brake_Curves.xlsx"
images_folder = r"C:\Users\Raghavendra\Downloads\WCR VB Braking Trials\DN Images"

insert_images_into_excel(excel_file, images_folder)


[OK] Inserted image into sheet: ARE_DN
[OK] Inserted image into sheet: KPZ_DN
[OK] Inserted image into sheet: IDG_DN
[OK] Inserted image into sheet: GKB_DN_LL
[OK] Inserted image into sheet: LBN_DN_LL
[OK] Inserted image into sheet: RWJ_DN_LL

✔ All available images inserted successfully!
